# 3. Depth: where is information compressed?

Shallow layers keep fine detail and edges; deep layers keep abstract semantics. Comparing
explanations layer by layer shows where the encoder compresses information (paper, Figure 1 and
Appendix I).

**What to look for:** the depth at which the scene stops being recognisable in the explanation map.

In [ ]:
# %pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

TRAINING = "imagenet"     # "random" to skip downloads
K = 8

In [ ]:
rows = {"single block": ["N1", "N2", "N3", "N4"], "cumulative": ["N4", "N34", "N234", "N1234"]}
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for r, (label, settings) in enumerate(rows.items()):
    for c, layers in enumerate(settings):
        ex = NAVEImage(model="resnet50", training=TRAINING, layers=layers, k=K)
        show(axes[r, c], ex, ex.explain(image), f"ResNet50 {layers}")
fig.tight_layout()

### Do concept boundaries follow image edges?

For each setting: the mean image-gradient magnitude on region borders, divided by its mean over the
whole image. Above 1, borders sit on image edges; shallow layers should score higher.

In [ ]:
gray = image.mean(2).astype(float)
gy, gx = np.gradient(gray)
grad = np.hypot(gx, gy)
for layers in ["N1", "N2", "N3", "N4", "N34", "N234", "N1234"]:
    exp = NAVEImage(model="resnet50", training=TRAINING, layers=layers, k=K).explain(image)
    border = exp.border(1) == 1
    print(f"{layers:6s} edge alignment {grad[border].mean() / grad.mean():.2f}   regions {exp.segmentation().max() + 1}")

### ViT blocks

A ViT keeps the same resolution at every block, so depth changes only what the tokens encode.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4.2))
for ax, b in zip(axes, [3, 6, 9, 12]):
    ex = NAVEImage(training=TRAINING, layers=[b], k=K)                 # ViT-Ti/16
    show(ax, ex, ex.explain(image), f"ViT-Ti/16 block {b}")

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@article{boubekki2024nave,
  title   = {Multi-Depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  author  = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  journal = {arXiv preprint arXiv:2411.19700},
  year    = {2024}
}
```